# Hackathon – Dataset C (COVID vaccine) · Notebook 05 of 09
## Modelling – six model families and hyper-parameter tuning

**Module:** 7PAM2002 – University of Hertfordshire · **GitHub:** `<add link>` · **Colab:** `<add link>` · **Author of this notebook:** `<name>`

**Goal of the project:** predict which residents will accept a COVID vaccine and decide who to post a dose to so that the town's **total
cost (vaccines + hospitalisations)** is as small as possible.

<details><summary><b>The nine notebooks (run them in this order)</b></summary>

| # | Notebook | Content |
|---|---|---|
| 01 | `01_Problem_Framing_and_Cost_Metric.ipynb` | Problem framing and the task-specific cost metric |
| 02 | `02_EDA_Data_Quality.ipynb` | EDA part 1 – overview, target, missing values, data quality, train/test shift |
| 03 | `03_EDA_Feature_Relationships.ipynb` | EDA part 2 – distributions, associations, interactions, cost drivers |
| 04 | `04_Preprocessing.ipynb` | Pre-processing – cleaning, feature engineering, comparing four options |
| 05 | `05_Model_Tuning.ipynb` | Modelling – six model families and hyper-parameter tuning |
| 06 | `06_Cross_Validation_and_Decision_Strategy.ipynb` | Repeated cross-validation and turning probabilities into decisions |
| 07 | `07_Holdout_Evaluation_and_Robustness.ipynb` | Unseen hold-out evaluation, bootstrap uncertainty, robustness to assumptions |
| 08 | `08_Final_Selection_and_Submission.ipynb` | Final model selection and the five ranked submission files |
| 09 | `09_Explainable_AI_and_Conclusions.ipynb` | Explainable AI (SHAP, odds ratios, permutation importance) and conclusions |

Shared code (data loading, cost metric, cleaning, pipelines, models, evaluation helpers) lives in **`hackathon_utils.py`** so every notebook uses
exactly the same definitions. Results are passed from one notebook to the next through the **`artifacts/`** folder.
</details>


In [1]:
# ---- Set-up: run from the repository folder locally, or in Google Colab ----
import sys, os, subprocess, importlib, importlib.util
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/<your-organisation>/<your-repository>.git'   # <- set this once for Colab

if IN_COLAB and not Path('hackathon_utils.py').exists():
    subprocess.run(['git', 'clone', '-q', REPO_URL, 'repo'], check=True)   # get code, data and artifacts
    os.chdir('repo')

for pkg in ['xgboost', 'lightgbm', 'catboost', 'shap']:                    # CatBoost is not pre-installed on Colab
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from hackathon_utils import *            # shared data, cost metric, pipelines, models and helpers
print('train:', train_raw.shape, '| test:', test_raw.shape, '| scikit-learn', sklearn.__version__)

train: (4756, 31) | test: (4749, 30) | scikit-learn 1.6.1


## 5. Modelling and hyper-parameter tuning
### 5.1 Candidate models
* **Logistic regression** (elastic-net): interpretable baseline, naturally well calibrated.
* **Random forest**: bagged trees, robust; its probabilities are known to be squeezed towards 0.5, so we also test an **isotonic-calibrated** version.
* **XGBoost**, **LightGBM**: gradient-boosted trees, capture the interactions seen in the EDA.
* **CatBoost**: boosting with *native* categorical handling (ordered target statistics) – no one-hot needed.
* **Soft-voting ensemble** of LR + XGBoost + LightGBM + CatBoost: averaging diverse models usually improves calibration and stability.

**Tuning objective.** Hyper-parameters are tuned with `RandomizedSearchCV` (5-fold) on **log-loss**. Our decisions depend on the
*probability* values (they are compared with personal thresholds), so we want the most accurate, calibrated probabilities; log-loss is a
proper scoring rule that rewards exactly that, and it is much smoother than the dollar cost (which jumps when a probability crosses a
threshold). The dollar cost is then used to **compare and select** models.

The CatBoost wrapper and the elastic-net logistic regression (from `hackathon_utils.py`):

In [2]:
show_source(CatBoostSK)
show_source(make_lr)

```python
class CatBoostSK(ClassifierMixin, BaseEstimator):
    """Small scikit-learn wrapper around CatBoost (native categorical features, clone-safe on all versions)."""

    def __init__(self, iterations=500, learning_rate=0.05, depth=5, l2_leaf_reg=3.0, thread_count=-1,
                 random_state=RANDOM_STATE):
        self.iterations, self.learning_rate, self.depth = iterations, learning_rate, depth
        self.l2_leaf_reg, self.thread_count, self.random_state = l2_leaf_reg, thread_count, random_state

    def _frame(self, X):
        X = X[CB_FEATURES].copy()
        X[CATEGORICAL] = X[CATEGORICAL].fillna('Missing').astype(str)
        return X

    def fit(self, X, y):
        self.model_ = CatBoostClassifier(iterations=self.iterations, learning_rate=self.learning_rate, depth=self.depth,
                                         l2_leaf_reg=self.l2_leaf_reg, thread_count=self.thread_count,
                                         random_seed=self.random_state, cat_features=CATEGORICAL, verbose=0,
                                         allow_writing_files=False)
        self.model_.fit(self._frame(X), np.asarray(y))
        self.classes_ = np.unique(y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(self._frame(X))

    def predict(self, X):
        return self.classes_[(self.predict_proba(X)[:, 1] >= 0.5).astype(int)]

    def shap_values(self, X):
        """Exact TreeSHAP values on the original (un-encoded) features."""
        Xf = self._frame(X)
        sv = self.model_.get_feature_importance(Pool(Xf, cat_features=CATEGORICAL), type='ShapValues')
        return sv[:, :-1], sv[:, -1], Xf

```

```python
def make_lr(C=0.1, l1_ratio=0.0):
    """Elastic-net logistic regression that works on old and new scikit-learn versions."""
    if SK_VERSION >= (1, 8):
        return LogisticRegression(C=C, l1_ratio=l1_ratio, solver='saga', max_iter=5000)
    return LogisticRegression(C=C, l1_ratio=l1_ratio, penalty='elasticnet', solver='saga', max_iter=5000)

```

In [3]:
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV

BEST_PREP = load_json('best_prep.json')          # chosen in notebook 04
print('Pre-processing per family:', BEST_PREP)
BASE = base_estimators(BEST_PREP)                 # un-tuned pipelines (see hackathon_utils.py)

# (pipeline, search space, number of random candidates)
SEARCH_SPACES = {
    'Logistic Regression': (BASE['Logistic Regression'],
                            {'clf__C': loguniform(1e-3, 10), 'clf__l1_ratio': uniform(0, 1)}, 15),
    'Random Forest': (BASE['Random Forest'],
                      {'clf__n_estimators': randint(200, 600), 'clf__max_depth': [None, 6, 8, 10, 14, 18],
                       'clf__min_samples_leaf': randint(1, 20), 'clf__max_features': ['sqrt', 0.2, 0.35, 0.5]}, 15),
    'XGBoost': (BASE['XGBoost'],
                {'clf__n_estimators': randint(150, 800), 'clf__learning_rate': loguniform(0.01, 0.2),
                 'clf__max_depth': randint(2, 7), 'clf__subsample': uniform(0.6, 0.4),
                 'clf__colsample_bytree': uniform(0.4, 0.6), 'clf__min_child_weight': randint(1, 12),
                 'clf__reg_lambda': loguniform(0.1, 20)}, 25),
    'LightGBM': (BASE['LightGBM'],
                 {'clf__n_estimators': randint(150, 800), 'clf__learning_rate': loguniform(0.01, 0.2),
                  'clf__num_leaves': randint(4, 48), 'clf__min_child_samples': randint(10, 100),
                  'clf__subsample': uniform(0.6, 0.4), 'clf__colsample_bytree': uniform(0.4, 0.6),
                  'clf__reg_lambda': loguniform(0.01, 20)}, 25),
    'CatBoost': (BASE['CatBoost'],
                 {'clf__iterations': randint(300, 900), 'clf__learning_rate': loguniform(0.02, 0.15),
                  'clf__depth': randint(3, 8), 'clf__l2_leaf_reg': loguniform(1, 20)}, 10),
}

TUNED_PARAMS, tuning_rows = {}, []
for name, (pipe, space, n_iter) in SEARCH_SPACES.items():
    t0 = time.time()
    search = RandomizedSearchCV(pipe, space, n_iter=n_iter, scoring='neg_log_loss', cv=TUNE_CV,
                                random_state=RANDOM_STATE, n_jobs=N_JOBS, refit=False)
    search.fit(X_dev, y_dev)
    TUNED_PARAMS[name] = search.best_params_
    res = pd.DataFrame(search.cv_results_)
    tuning_rows.append({'model': name, 'n_candidates': n_iter, 'best CV log-loss': -search.best_score_,
                        'worst CV log-loss': -res['mean_test_score'].min(), 'seconds': round(time.time() - t0),
                        'best params': {k.replace('clf__', ''): (round(v, 4) if isinstance(v, float) else v)
                                        for k, v in search.best_params_.items()}})
    print(f'{name:20s} best log-loss {-search.best_score_:.4f}  ({time.time() - t0:.0f}s)')

tuning_table = pd.DataFrame(tuning_rows).set_index('model')

# ---- Save for the next notebooks ----
save_json(TUNED_PARAMS, 'tuned_params.json')
tuning_table.to_csv(ART_DIR / 'tuning_table.csv')
tuning_table

Pre-processing per family: {'LogReg': 'P4_binned', 'LightGBM': 'P3_ordinal_fe'}
Logistic Regression  best log-loss 0.4261  (7s)
Random Forest        best log-loss 0.4316  (4s)
XGBoost              best log-loss 0.4184  (1s)
LightGBM             best log-loss 0.4191  (2s)
CatBoost             best log-loss 0.4197  (11s)


,n_candidates,best CV log-loss,worst CV log-loss,seconds,best params
model,,,,,
Logistic Regression,15,0.426083,0.632154,7,"{'C': 0.2538, 'l1_ratio': 0.7081}"
Random Forest,15,0.431570,0.467261,4,"{'max_depth': 10, 'max_features': 0.35, 'min_s..."
XGBoost,25,0.418439,0.551181,1,"{'colsample_bytree': 0.4198, 'learning_rate': ..."
LightGBM,25,0.419125,0.617353,2,"{'colsample_bytree': 0.4003, 'learning_rate': ..."
CatBoost,10,0.419723,0.442278,11,"{'depth': 7, 'iterations': 643, 'l2_leaf_reg':..."


### 5.2 Models built on top of the tuned ones
Besides the five tuned models we evaluate three more candidates (created by `build_tuned_models` in `hackathon_utils.py`):
* **Random Forest + isotonic calibration** – tests whether better-calibrated probabilities lower the cost.
* **Soft-voting ensemble** of the tuned logistic regression, XGBoost, LightGBM and CatBoost (equal weights).
* **Prior-only baseline** – predicts the same acceptance probability (≈0.33) for everybody.

**Next:** notebook 06 compares all of them with repeated cross-validation using the cost metric.

In [4]:
show_source(build_tuned_models)

```python
def build_tuned_models(best_prep, tuned_params):
    """Re-create every candidate model from the tuned hyper-parameters saved by notebook 05."""
    models = {}
    for name, est in base_estimators(best_prep).items():
        models[name] = est.set_params(**tuned_params[name])
    models['CatBoost'].set_params(clf__thread_count=-1)            # all cores outside the parallel search
    rf_best = models['Random Forest'].named_steps['clf']
    models['Random Forest + isotonic'] = make_pipe(best_prep['LightGBM'],
                                                   CalibratedClassifierCV(clone(rf_best), method='isotonic', cv=5))
    models['Ensemble (LR+XGB+LGBM+Cat)'] = VotingClassifier(
        [('lr', models['Logistic Regression']), ('xgb', models['XGBoost']), ('lgbm', models['LightGBM']),
         ('cat', models['CatBoost'])], voting='soft')
    models['Baseline: prior only'] = DummyClassifier(strategy='prior')
    return {k: models[k] for k in MODEL_ORDER}

```